In [ ]:
KEY = ...

In [28]:
import requests
from pprint import pprint
import typing as t

# 1. Создание моделей данных

In [29]:
from pydantic import BaseModel, Field, AliasPath, BeforeValidator

In [30]:
import re

In [31]:
class Route(BaseModel):
    distance: int
    duration: int

In [32]:
class Point(BaseModel):
    lat: float
    lon: float

In [33]:
def collect_rubrics(rubrics: dict[str, t.Any]) -> list[str]:
    return [rubric['name'] for rubric in rubrics]


def get_avg_lunch_cost(stop_factors: list[dict[str, str]]) -> int:
    for stop_factor in stop_factors:
        if stop_factor.get('tag') == 'food_service_avg_price':
            return int(re.search('\d+', stop_factor.get('name')).group(0))
    return -1


def get_avg_buisness_lunch_cost(stop_factors: list[dict[str, str]]) -> int:
    for stop_factor in stop_factors:
        if stop_factor.get('tag') == 'food_service_lunch_cost':
            return int(re.search('\d+', stop_factor.get('name')).group(0))
    return -1


def get_cuisines(stop_factors: list[dict[str, str]]) -> list[str]:
    cuisines = []
    for stop_factor in stop_factors:
        if 'food_service_food_' in stop_factor.get('tag', ''):
            cuisines.append(stop_factor.get('name'))
    return cuisines

In [34]:
class PlaceInfo(BaseModel):
    place_id: str = Field(validation_alias='id')
    address_name: str
    place_name: str = Field(validation_alias='name')
    point: Point
    general_rating: float = Field(validation_alias=AliasPath('reviews', 'general_rating'))
    rubrics: t.Annotated[list[str], BeforeValidator(collect_rubrics)]
    avg_lunch_cost: t.Annotated[int, BeforeValidator(get_avg_lunch_cost)] = Field(-1, validation_alias=AliasPath('context', 'stop_factors'))
    avg_buisness_lunch_cost: t.Annotated[int, BeforeValidator(get_avg_buisness_lunch_cost)] = Field(-1, validation_alias=AliasPath('context', 'stop_factors'))
    cuisines: t.Annotated[list[str], BeforeValidator(get_cuisines)] = Field(-1, validation_alias=AliasPath('context', 'stop_factors'))

# 2. Класс с методами для обращения к API

In [35]:
import traceback

In [36]:
import json

In [37]:
class Map2GisAPI:
    @staticmethod
    def get_coords_by_address(address: str) -> Point | None:
        """Метод для получения долготы и широты объекта по его физическому адресу

        Args:
            address (str): Текстовое значение адреса, состоящего из города, улицы и номера дома

        Returns:
            Point | None: Значение широты и долготы в случае, если запрос прошел, иначе -- None

        Examples:
            >>> Map2GisAPI.get_coords_by_address('Санкт-Петербург, Виленский переулок, 14')
            Point('lat': 59.94028, 'lon': 30.369012)
        """
        url = 'https://catalog.api.2gis.com/3.0/items/geocode'
        params = {
            'q': address,
            'fields': 'items.point',
            'key': KEY
        }
        try:
            response = requests.get(url, params=params)
            return Point(**response.json()['result']['items'][0]['point'])
        except:
            return None

    @staticmethod
    def get_places_info(lat: float, lon: float, query: str = 'обед с бизнес-ланчем', radius: int = 1000) -> list[PlaceInfo]:
        """Метод для получения информации об объектах в радиусе по данной широте и долготе

        Args:
            lat (float): Широта точки
            lon (float): Долгота точки
            querry (float): Запрос, по которому осуществляется поиск
            radius (float): Радиус (в метрах), по которому осуществляется поиск

        Returns:
            list[PlaceInfo]: Список информации о местах общественого питания, найденный в данном радиусе относительно точки

        Examples:
            >>> Map2GisAPI.get_places_info(lat=59.94028, lon=30.369012)
            [PlaceInfo(...), ..., PlaceInfo(...)]
        """
        url = 'https://catalog.api.2gis.com/3.0/items'
        page_size = 10
        places = []
        params = {
            'q': query,
            'lon': lon,
            'lat': lat,
            'radius': radius,
            'type': 'branch',
            'fields': 'items.point,items.rubrics,items.description,items.reviews,items.statistics,items.context',
            'page': 1,
            'page_size': page_size,
            'key': KEY
        }

        try:
            while True:
                response = requests.get(url, params=params)
                response.raise_for_status()
                data = response.json().get('result', {})

                if not data:
                    break

                places.extend([place for place in data.get('items', [])])

                total = data.get('total', 0)
                if len(places) >= total:
                    break

                params['page'] += 1

        except Exception as e:
            print("Произошла ошибка:")
            traceback.print_exc()

        return [PlaceInfo(**place) for place in places]


    @staticmethod
    def get_place_info(address: str, place_name: str) -> PlaceInfo | None:
        """Метод для получения информации об объекте по его адресу и названию

        Args:
            address (str): Текстовое значение адреса, состоящего из города, улицы и номера дома
            place_name (str): Название заведения

        Returns:
            PlaceInfo | None: Информация о заведении

        Examples:
            >>> Map2GisAPI.get_place_info(address='Омск, проспект Мира 9', place_name='Ланч-Тайм')
            PlaceInfo(...)
        """
        url = 'https://catalog.api.2gis.com/3.0/items'

        params = {
            'q': f'{address}, {place_name}',
            'type': 'branch',
            'fields': 'items.point,items.rubrics,items.description,items.reviews,items.statistics,items.context',
            'key': KEY
        }

        try:
            response = requests.get(url, params=params)
            return PlaceInfo(**response.json()['result']['items'][0])
        except:
            return None

    @staticmethod
    def get_route(from_point: Point, to_point: Point) -> Route | None:
        """Метод для получения расстояния от точки до точки и времени пути

        Args:
            from_point (Point): Начальная точка
            to_point (Point): Конечная точка

        Returns:
            Route | None: Информация о маршруте

        Examples:
            >>> Map2GisAPI.get_route(Point(lat=59.942208, lon=30.355653), Point(lat=59.94156, lon=30.363407))
            Route(distance=536, duration=357)
        """
        url = 'https://routing.api.2gis.com/get_dist_matrix'

        headers = {
            'Content-Type': 'application/json'
        }

        data = {
            'points': [
                dict(from_point),
                dict(to_point)
            ],
            'sources': [0],
            'targets': [1],
            'transport': 'walking',
            'type': 'shortest',
        }

        params = {
            'key': KEY
        }

        try:
            response = requests.post(url, data=json.dumps(data), headers=headers, params=params)
            return Route(**response.json()['routes'][0])
        except:
            return None

# 3. Примеры работы функций

In [38]:
Map2GisAPI.get_coords_by_address('Санкт-Петербург, Виленский переулок, 14')

Point(lat=59.94028, lon=30.369012)

In [39]:
Map2GisAPI.get_places_info(lat=59.94028, lon=30.369012)

[PlaceInfo(place_id='70000001069595862', address_name='Маяковского, 39', place_name='У Ларисы, кафе-бар', point=Point(lat=59.942208, lon=30.355653), general_rating=4.6, rubrics=['Кафе', 'Бары', 'Доставка еды', 'Рюмочные'], avg_lunch_cost=-1, avg_buisness_lunch_cost=-1, cuisines=['Узбекская кухня']),
 PlaceInfo(place_id='5348553838529810', address_name='Радищева, 36', place_name='Траппист, бельгийская брассерия', point=Point(lat=59.94156, lon=30.363407), general_rating=4.8, rubrics=['Рестораны', 'Бары', 'Доставка еды'], avg_lunch_cost=2000, avg_buisness_lunch_cost=990, cuisines=['Французская кухня']),
 PlaceInfo(place_id='70000001093839054', address_name='Восстания, 55', place_name='Мама Тата, грузинская неорюмочная', point=Point(lat=59.943178, lon=30.360953), general_rating=4.7, rubrics=['Кафе', 'Рюмочные', 'Бары'], avg_lunch_cost=850, avg_buisness_lunch_cost=350, cuisines=['Грузинская кухня', 'Кавказская кухня', 'Европейская кухня']),
 PlaceInfo(place_id='70000001060749771', address_n

In [40]:
Map2GisAPI.get_place_info(address='Омск, проспект Мира 9', place_name='Ланч-Тайм')

PlaceInfo(place_id='70000001059833071', address_name='проспект Мира, 9Б', place_name='Ланч-Тайм, сеть столовых', point=Point(lat=55.024966, lon=73.294175), general_rating=4.2, rubrics=['Столовые'], avg_lunch_cost=249, avg_buisness_lunch_cost=-1, cuisines=[])

In [41]:
Map2GisAPI.get_route(Point(lat=59.942208, lon=30.355653), Point(lat=59.94156, lon=30.363407))

Route(distance=536, duration=357)